# Binder / Environment Setup
## Requirements
numpy
pandas
scipy
matplotlib
seaborn
astral
phreeqpy
iphreeqc

In [ ]:
# If running in a fresh Binder environment
# !pip install numpy pandas scipy matplotlib seaborn astral phreeqpy

In [3]:
# Imports
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from astral.sun import sun
from astral import LocationInfo

# from phreeqpy.iphreeqc.phreeqc import IPhreeqc
import phreeqpy.iphreeqc.phreeqc_dll as phreeqc_mod

In [4]:
# ---------------- USER PARAMETERS ---------------- #

LATITUDE = 35.939311                  # Nashville region
DATE = "2025-07-18"
LAKE_DEPTH_METERS = 1.0

ATM_d13C_CO2 = -8.5              # ‰ VPDB
ORG_d13C = -28.0                 # ‰ organic matter

PF_INIT = -20.0                  # photosynthetic fractionation
K600_INIT = 1.0                  # m d-1

CSV_FILE = "SL-20250718.csv"

In [8]:
# Load & Validate Data
df = pd.read_csv(CSV_FILE)

# Combine date and time
# df["DateTime"] = pd.to_datetime(df["DATE"] + " " + df["TIME"])
# df = df.sort_values("DateTime").reset_index(drop=True)

# Rename for clarity
df = df.rename(columns={
    "d13C-DIC (‰)": "d13C_obs",
    "DIC (mg L-1) ": "DIC",
    "DO (mg L-1)": "DO",
    "Temperature (°C)": "TempC",
    "GPP (mg O2 L-1 h-1)": "GPP",
    "R (mg O2 L-1 h-1)": "ER",
    "KO2 (m d-1)": "K02"
})

# Convert ER to positive carbon production
df["ER"] = np.abs(df["ER"])

In [14]:
# Sunrise / Sunset & Daylight Flag
loc = LocationInfo(latitude=LATITUDE, longitude= -87.015833)
sun_info = sun(loc.observer, date=pd.to_datetime(DATE))

# This part returns an error "TypeError: '<=' not supported between instances of 'datetime.datetime' and 'str'"
df["is_day"] = df["DateTime"].apply(
    lambda t: sun_info["sunrise"] <= t <= sun_info["sunset"]
).astype(int)

df.loc[df["is_day"] == 0, "GPP"] = 0.0

TypeError: '<=' not supported between instances of 'datetime.datetime' and 'str'

In [13]:
print (sun_info["sunset"])

2025-07-18 01:03:19.657494+00:00


In [ ]:
# Inline PHREEQC Carbon–Isotope Template
PHREEQC_TEMPLATE = """
SOLUTION 1
    temp {temp}
    pH {pH}
    units mg/L
    C {dic}
    -water 1.0

ISOTOPE
    13C {d13c}

SELECTED_OUTPUT
    -reset false
    -pH
    -totals C
    -molalities CO2 HCO3- CO3-2
END
"""


In [ ]:
# PHREEQC Execution Wrapper
# iph = IPhreeqc()
iph = phreeqc_mod()
iph.load_database("phreeqc.dat")

def run_phreeqc(temp, pH, dic, d13c):
    iph.run_string(PHREEQC_TEMPLATE.format(
        temp=temp,
        pH=pH,
        dic=dic,
        d13c=d13c
    ))
    out = iph.get_selected_output_array()
    headers = out[0]
    values = out[1]
    return dict(zip(headers, values))

In [ ]:
# Gas Exchange + Isotopes
def gas_flux(d13c, dic, k600, depth):
    eq_d13c = ATM_d13C_CO2 + 1.0
    flux = k600 * (dic - 2.0) / depth
    d13c_new = (dic*d13c - flux*(d13c-eq_d13c)) / dic
    return d13c_new

In [ ]:
# GPP & ER Isotope Effects
def apply_gpp(d13c, dic, gpp, pf):
    uptake = gpp * 0.375
    return (dic*d13c - uptake*(d13c+pf)) / dic

def apply_er(d13c, dic, er):
    return (dic*d13c + er*ORG_d13C) / dic

In [ ]:
# Forward time integration
def simulate(params):
    pf, k600 = params
    d13c = df.loc[0, "d13C_obs"]
    modeled = []

    for _, r in df.iterrows():
        d13c = apply_er(d13c, r["DIC"], r["ER"])
        if r["is_day"]:
            d13c = apply_gpp(d13c, r["DIC"], r["GPP"], pf)
        d13c = gas_flux(d13c, r["DIC"], k600, LAKE_DEPTH_METERS)
        modeled.append(d13c)

    return np.array(modeled)

In [ ]:
# Optimization Sequence (PF + k600)
def objective(x):
    mod = simulate(x)
    return np.sqrt(np.mean((mod - df["d13C_obs"])**2))

res = minimize(
    objective,
    x0=[PF_INIT, K600_INIT],
    bounds=[(-30, -15), (0.5, 5.0)]
)

PF_OPT, K600_OPT = res.x
df["d13C_model"] = simulate(res.x)
df["residual"] = df["d13C_model"] - df["d13C_obs"]

In [ ]:
RMSE = np.sqrt(np.mean(df["residual"]**2))
R2 = np.corrcoef(df["d13C_model"], df["d13C_obs"])[0,1]**2
``

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df["DateTime"], df["d13C_obs"], 'k', label="Observed")
plt.plot(df["DateTime"], df["d13C_model"], 'r', label="Modeled")
plt.fill_between(df["DateTime"], -15, 0,
                 where=df["is_day"]==0,
                 alpha=0.2, color="gray")
plt.ylabel("δ¹³C-DIC (‰ VPDB)")
plt.legend()
plt.grid()
plt.title(f"RMSE={RMSE:.2f}‰   R²={R2:.2f}")
plt.savefig("d13C_DIC_model.svg")
plt.show()

In [ ]:
# CSV export
df_out = df.copy()
df_out["PF_opt"] = PF_OPT
df_out["k600_opt"] = K600_OPT
df_out.to_csv("diel_carbon_isotope_model_output.csv", index=False)